# Mouse brain extraction, parcellation and cortical thickness

Combine T2 mouse extraction with the Nick parcellation and Kelly–Kapowski
thickness workflow. **Resources:** downloads the tutorial mouse T2 acquisition, atlas data
and model weights; registration and thickness estimation can be slow and
memory intensive. Default CPU; select an accelerator only if supported.
The default image is the acquired mouse T2 used in the ANTsX tutorial;
set t2_path to use a different local subject.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

In [ ]:
from pathlib import Path
t2_path = None  # Or an absolute path to a scalar 3-D mouse T2 MRI.
if t2_path is None:
    t2_path = antstorch.get_antstorch_data("tutorialMouseT2")
t2 = ants.image_read(str(t2_path)).clone("float")
assert t2.dimension == 3 and not t2.has_components
assert np.isfinite(t2.numpy()).all()
print("Input:", t2_path, "shape:", t2.shape, "spacing:", t2.spacing)
probability = antstorch.mouse_brain_extraction(
    t2, modality="t2", return_isotropic_output=False, device=device, verbose=True,
)
assert probability.shape == t2.shape and np.isfinite(probability.numpy()).all()
mask = ants.threshold_image(probability, 0.5, 1., 1, 0)
assert (mask.numpy() > 0).any()
ants.plot(t2, overlay=mask, axis=2, nslices=6,
          overlay_cmap="Reds", overlay_alpha=0.5)

## Parcellation and thickness together
`mouse_cortical_thickness` internally calls `mouse_brain_parcellation` with
`which_parcellation="nick"`; retrieve that parcellation from the returned
dictionary rather than running the same network twice. The mask is reused.
Computation uses an isotropic intermediate grid. Returning that grid explicitly
avoids treating it as identical to the native subject grid.

In [ ]:
result = antstorch.mouse_cortical_thickness(
    t2, mask=mask, return_isotropic_output=True,
    backend="antstorch", device=device, verbose=True,
)
parcellation = result["parcellation"]
segmentation = parcellation["segmentation_image"]
thickness = result["thickness_image"]
assert thickness.shape == segmentation.shape
assert np.isin(segmentation.numpy(), np.arange(7)).all()
assert np.isfinite(thickness.numpy()).all() and (thickness.numpy() >= 0).all()
assert (thickness.numpy() > 0).any()
for attribute in ("spacing", "origin", "direction"):
    np.testing.assert_allclose(getattr(thickness, attribute), getattr(segmentation, attribute))
# Resample the anatomical image only for display on the output grid.
display_t2 = ants.resample_image_to_target(t2, segmentation, interp_type="linear")
ants.plot(display_t2, overlay=segmentation, axis=2, nslices=6,
          overlay_cmap="jet", overlay_alpha=0.5)
ants.plot(display_t2, overlay=thickness, axis=2, nslices=6,
          overlay_cmap="Reds", overlay_alpha=0.6)

In [ ]:
names = ["background", "cerebral cortex", "cerebral nuclei", "brain stem",
         "cerebellum", "main olfactory bulb", "hippocampal formation"]
voxel_volume = float(np.prod(segmentation.spacing))
for label, name in enumerate(names[1:], start=1):
    volume = np.count_nonzero(segmentation.numpy() == label) * voxel_volume
    print(f"{label}: {name}: {volume:.4f} cubic physical units")
output_dir = Path("example_outputs/mouse_brain_analysis")
output_dir.mkdir(parents=True, exist_ok=True)
for name, image in {"brain_probability_native": probability, "brain_mask_native": mask,
                    "parcellation_isotropic": segmentation, "thickness_isotropic": thickness}.items():
    ants.image_write(image, str(output_dir / f"{name}.nii.gz"))
print("Outputs (matching files are overwritten):", output_dir.resolve())

Native-space and isotropic files are named separately because their grids
differ. Thickness uses the physical units of the image spacing, typically mm;
verify those units for mouse data. The thickness wrapper groups cortical and
hippocampal tissue according to its implementation; do not interpret it as a
human-style cortical ribbon without examining the tissue definitions.